In [2]:
import os
import requests
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv()
os.environ["GROQ_API_KEY"] =  os.getenv("GROQ_API_KEY")
WEATHER_API_KEY =  os.getenv("WEATHER_API_KEY")

@tool("weather_search")
def get_weather_update(city:str) -> str:
    """this tool returns the weather of an asked city.
    Args: 
        city: name of the city to search the weather for
    """
    url = f"https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city, 
    "appid": WEATHER_API_KEY,
    "units": "metric"
    }
    response = requests.get(url, params = params, timeout=10)

    response.raise_for_status()

    data = response.json()
    
    return (
        f"Weather in {data['name']}: "
        f"{data['main']['temp']}°C, "
        f"Condition: {data['weather'][0]['description']}, "
        f"Humidity: {data['main']['humidity']}%"
    )



model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("how are you")
response.content


"I'm doing well, thank you for asking! How are you doing today? Is there anything specific I can help you with?"

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[get_weather_update]
)
respones = agent.invoke({"messages": [{"role": "user", "content": "What will be the weather of Bareilly Uttar Pradesh at night today"}]})
respones["messages"][-1].content


'Based on the current weather data for Bareilly, Uttar Pradesh:\n\n*   **Temperature:** 23.03°C\n*   **Condition:** Clear sky\n*   **Humidity:** 30%\n\nSince it is currently clear with low humidity, the night is likely to be **pleasant and dry**. Temperatures may drop slightly as the sun sets, but you can generally expect a comfortable evening without rain or cloud cover.'